In [0]:
# CELL 1
from pyspark.sql import functions as F

df = spark.read.table("sarkarimitracatalog.sarkaridatabase.schemes_structured") \
    .select("slug", "scheme_name", "applicable_states", "category_eligibility",
            "gender_eligibility", "age_min", "age_max",
            "income_limit_inr", "schemeCategory")

print(f"✅ Loaded {df.count()} schemes")

✅ Loaded 3399 schemes


In [0]:
# CELL 2 — Find overlapping scheme pairs
df_overlap = df.alias("a").join(
    df.alias("b"),
    on=(
        (F.col("a.slug") < F.col("b.slug")) &
        (F.col("a.applicable_states") == F.col("b.applicable_states")) &
        (F.col("a.category_eligibility") == F.col("b.category_eligibility")) &
        (F.col("a.age_min") <= F.col("b.age_max")) &
        (F.col("a.age_max") >= F.col("b.age_min"))
    ),
    how="inner"
).select(
    F.col("a.slug").alias("scheme_slug_a"),
    F.col("b.slug").alias("scheme_slug_b"),
    F.col("a.scheme_name").alias("scheme_name_a"),
    F.col("b.scheme_name").alias("scheme_name_b"),
    F.col("a.schemeCategory").alias("category"),
    F.col("a.applicable_states").alias("shared_state")
)

print(f"✅ Overlap pairs found: {df_overlap.count()}")
display(df_overlap.limit(20))

✅ Overlap pairs found: 200966


scheme_slug_a,scheme_slug_b,scheme_name_a,scheme_name_b,category,shared_state
fa,fsme,Funeral Assistance,First Self Marriage Expenses,Social welfare & Empowerment,"[""Maharashtra""]"
faabocwwb,fss,Funeral Assistance (A.B.O.C.W.W.B),Film Finance Scheme,Social welfare & Empowerment,"[""Assam""]"
facbocwwb,fapmdcbcbocwwb,Funeral Assistance (CBOCWWB),Financial Assistance to Physically/Mentally Disabled Children of Beneficiary (CBOCWWB),Social welfare & Empowerment,"[""Punjab""]"
fa-gbocwwb,fimdgc,Funeral Assistance (GBOCWWB),Financial Incentives To Mothers Who Deliver A Girl Child (Mamta),Social welfare & Empowerment,"[""Goa""]"
fahbocwwb,freepassport,Funeral Assistance (HBOCWWB),Free Passport Scheme,Social welfare & Empowerment,"[""Haryana""]"
fasgbocwwb,ftstbdp,Funeral Assistance Scheme (GBOCWWB),Free Travel in S.T Bus to the Disabled Persons,Social welfare & Empowerment,"[""Gujarat""]"
fasmp,fdotbts,Funeral Assistance Scheme (Madhya Pradesh),Free Distribution of Text Books to Students,"Health & Wellness, Social welfare & Empowerment","[""Madhya Pradesh""]"
fas-pbaocwwb,fsrpp,Funeral Assistance Scheme (PBAOCWWB),Free Supply of Rice to the Poor People,Social welfare & Empowerment,"[""Puducherry""]"
fas-pulws,fsrpp,Funeral Assistance Scheme (PULWS),Free Supply of Rice to the Poor People,Social welfare & Empowerment,"[""Puducherry""]"
facw,fsotb,Funeral Assistance for the Construction Workers,Free Supply of Text Books,Social welfare & Empowerment,"[""Delhi""]"


In [0]:
# CELL 3 — Write
df_overlap.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sarkarimitracatalog.sarkaridatabase.scheme_document_overlap")

print("✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_document_overlap")

✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_document_overlap
